# Halifax Digital Twin - Custom Simulation

Interactive sandbox for testing custom radio scenarios in the Halifax peninsula Sionna scene.
Use this notebook after the demo notebook when you want to change transmitter selection,
custom TX placement, receiver-plane elevation, antenna pattern, transmit power, or ray-tracing settings.

**Real-antenna mode** - When you select a frequency band, the notebook automatically
loads real antenna-sector rows from the Halifax antenna dataset for that band
(Bell Mobility, Rogers, Bragg). You can also define custom antennas.

**CPU/GPU note** - Sionna RT is much faster with a working NVIDIA CUDA/OptiX backend.
CPU mode is useful for verification, but keep TX count and rays per TX modest.

### Table of contents
1. [Setup](#setup)
2. [Scene loading](#scene)
3. [Carrier frequency](#freq)
4. [TX z override](#height)
5. [Transmit power override](#power)
6. [Antenna pattern](#pattern)
7. [Radio-map plane and ray tracing](#rt)
8. [Custom antennas](#custom)
9. [Run simulation](#run)
10. [Coverage map results](#results)
11. [3D scene preview](#preview)
12. [3D ray tracing](#raytrace3d)


## 1. Setup <a id='setup'></a>

Run this cell first. Loads all dependencies, reads the real antenna dataset,
builds the ITU-R P.527-3 wet-ground interpolator, and defines helper functions
used throughout the notebook.


In [ ]:
import os, sys, json, math, warnings, gc

# Runtime backend. Change USE_GPU, then restart the kernel and run all cells.
# The backend must be selected before importing Mitsuba, Sionna, Dr.Jit, or Torch.
USE_GPU = False  # True = GPU/CUDA, False = CPU/LLVM

if USE_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["MITSUBA_VARIANT"] = "cuda_ad_mono_polarized"
    print("Runtime mode: GPU")
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    os.environ["MITSUBA_VARIANT"] = "llvm_ad_mono_polarized"
    print("Runtime mode: CPU")

print("CUDA_VISIBLE_DEVICES =", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
print("MITSUBA_VARIANT =", os.environ.get("MITSUBA_VARIANT"))

import mitsuba as mi

_mitsuba_variant = os.environ["MITSUBA_VARIANT"]
try:
    mi.set_variant(_mitsuba_variant)
except Exception as exc:
    active_variant = None
    try:
        active_variant = mi.variant()
    except Exception:
        pass
    if active_variant != _mitsuba_variant:
        raise RuntimeError(
            f"Could not initialize Mitsuba variant '{_mitsuba_variant}'.\n"
            "Recommended on Jetson/NVIDIA GPU: cuda_ad_mono_polarized.\n"
            "CPU fallback: llvm_ad_mono_polarized. Windows CPU mode also needs LLVM."
        ) from exc

print(f"Sionna/Mitsuba backend: {mi.variant()}")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from pyproj import Transformer
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

import sionna
from sionna.rt import (load_scene, Transmitter, Receiver,
                       PlanarArray, RadioMaterial, RadioMapSolver, PathSolver)

from scipy.interpolate import make_smoothing_spline
import pyproj
from shapely.geometry import shape, Point
from shapely.ops import transform as shp_transform
try:
    from shapely.vectorized import contains as shp_contains
except ImportError:
    shp_contains = None

warnings.filterwarnings("ignore")

# Project paths and current Halifax inputs

def find_project_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root containing data")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
DATA_DIR = DATA_ROOT / "processed_data"
SCENE_DIR = DATA_ROOT / "scenes" / "halifax_peninsula"
OUT_DIR = PROJECT_ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENE_XML_MULTI = SCENE_DIR / "halifax_peninsula_multi_material.xml"
SCENE_XML_MONO_CONCRETE = SCENE_DIR / "halifax_peninsula_mono_concrete.xml"
SCENE_XML = str(SCENE_XML_MULTI)
ANTENNAS_CSV = DATA_DIR / "peninsula_cellular_antennas_2g_to_5g.csv"
BUILDING_HEIGHTS_GPKG = DATA_DIR / "building_heights_selected.gpkg"
OUT = str(OUT_DIR)

PENINSULA_COORDS = [
    (-63.6194204, 44.6410835),
    (-63.6308541, 44.6643012),
    (-63.6220173, 44.6817374),
    (-63.5992474, 44.6736948),
    (-63.5531831, 44.6413834),
    (-63.5565099, 44.6169733),
    (-63.5641958, 44.6133803),
    (-63.5844221, 44.6253175),
    (-63.6194298, 44.6410626),
    (-63.6194204, 44.6410835),
]
PENINSULA_FEATURE_COLLECTION = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {},
        "geometry": {"type": "Polygon", "coordinates": [PENINSULA_COORDS]},
    }],
}


def read_terrain_metadata(path):
    metadata = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if ":" not in line:
                continue
            key, value = line.strip().split(":", 1)
            value = value.strip()
            try:
                metadata[key] = float(value)
            except ValueError:
                metadata[key] = value
    return metadata

terrain_meta = read_terrain_metadata(SCENE_DIR / "terrain_metadata.txt")
X_ORI = float(terrain_meta["local_origin_x_utm_m"])
Y_ORI = float(terrain_meta["local_origin_y_utm_m"])
X_MAX = float(terrain_meta["local_max_x_m"])
Y_MAX = float(terrain_meta["local_max_y_m"])
CM_SIZE_X = float(X_MAX)
CM_SIZE_Y = float(Y_MAX)

def load_antenna_dataset(path):
    raw = pd.read_csv(path)
    tr = Transformer.from_crs("EPSG:4326", "EPSG:32620", always_xy=True)
    lon = pd.to_numeric(raw["LONGITUDE"], errors="coerce")
    lat = pd.to_numeric(raw["LATITUDE"], errors="coerce")
    x_utm, y_utm = tr.transform(lon.to_numpy(), lat.to_numpy())

    ground = pd.to_numeric(raw.get("DEM_GROUND_ELEV_M"), errors="coerce")
    ant_z = pd.to_numeric(raw.get("ANTENNA_Z_M"), errors="coerce")
    ant_height = pd.to_numeric(raw.get("TX_ANT_HT"), errors="coerce")
    z = ant_z.fillna(ground + ant_height).fillna(30.0)

    df = pd.DataFrame({
        "lat": lat,
        "lon": lon,
        "x_local_m": x_utm - X_ORI,
        "y_local_m": y_utm - Y_ORI,
        "z_local_m": z,
        "freq_mhz": pd.to_numeric(raw["TRANSMIT_FREQ"], errors="coerce"),
        "azimuth_deg": pd.to_numeric(raw.get("TX_ANT_AZIM"), errors="coerce").fillna(0.0),
        "tilt_deg": pd.to_numeric(raw.get("TX_ANT_ELEV_ANGLE"), errors="coerce").fillna(0.0),
        "power_dbm": pd.to_numeric(raw.get("TX_PWR"), errors="coerce"),
        "licensee": raw.get("LICENSEE", "").fillna(""),
        "service": raw.get("SERVICE", "").fillna(""),
        "technology": raw.get("TECHNOLOGY", "").fillna(""),
        "location": raw.get("LOCATION", "").fillna(""),
    })
    df = df.dropna(subset=["lat", "lon", "x_local_m", "y_local_m", "z_local_m", "freq_mhz"]).copy()
    df = df[(df["x_local_m"] >= 0) & (df["x_local_m"] <= X_MAX) &
            (df["y_local_m"] >= 0) & (df["y_local_m"] <= Y_MAX)].reset_index(drop=True)
    return df

df = load_antenna_dataset(ANTENNAS_CSV)

def get_peninsula_shape_local():
    pen = shape(PENINSULA_FEATURE_COLLECTION["features"][0]["geometry"])
    tr = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:32620", always_xy=True)
    pen = shp_transform(lambda x, y: tr.transform(x, y), pen)
    return shp_transform(lambda x, y: (x - X_ORI, y - Y_ORI), pen)

for name, file_path in [("SCENE_XML", SCENE_XML), ("ANTENNAS_CSV", ANTENNAS_CSV)]:
    ok = os.path.exists(file_path)
    print(f"  {'OK' if ok else 'X '} {name:20s} = {file_path}")
print(f"\nScene extent : {CM_SIZE_X:.0f} m (E) x {CM_SIZE_Y:.0f} m (N)")
print(f"Antennas     : {len(df)} sectors")

def build_geo_mask(H, W):
    pen = get_peninsula_shape_local()
    xx, yy = np.meshgrid(np.linspace(0, CM_SIZE_X, W), np.linspace(0, CM_SIZE_Y, H))
    if shp_contains:
        return shp_contains(pen, xx.ravel(), yy.ravel()).reshape(H, W)
    return np.array([pen.contains(Point(xi, yi))
                     for xi, yi in zip(xx.ravel(), yy.ravel())]).reshape(H, W)

def latlon_to_local(lat, lon):
    tr = Transformer.from_crs("EPSG:4326", "EPSG:32620", always_xy=True)
    x_utm, y_utm = tr.transform(lon, lat)
    return x_utm - X_ORI, y_utm - Y_ORI

def get_tx_power_dbm(z_m):
    
    if z_m >= 20: return 43.0   # macro cell
    if z_m >= 10: return 38.0   # mid-range
    return 24.0                 # small cell

# ITU-R P.527-3 interpolator — curve B, digitised from ITU Figure 1
# Columns: [frequency_MHz, relative_permittivity_er, conductivity_S_per_m]
_P527_B = np.array([
    [1e0,30.0,1.0e-2],[3e0,30.0,1.0e-2],[1e1,30.0,1.1e-2],[3e1,29.5,1.3e-2],
    [1e2,28.0,2.0e-2],[3e2,25.0,5.0e-2],[1e3,20.0,1.5e-1],[3e3,15.0,5.0e-1],
    [1e4,10.0,1.5e0 ],[3e4, 7.0,5.0e0 ],[1e5, 5.0,1.5e1 ],
])

_spl_er  = make_smoothing_spline(np.log10(_P527_B[:,0]), np.log10(_P527_B[:,1]), lam=1e-3)
_spl_sig = make_smoothing_spline(np.log10(_P527_B[:,0]), np.log10(_P527_B[:,2]), lam=1e-3)
def p527_wet(freq_hz):
    lf = np.log10(freq_hz / 1e6)   
    return float(10**_spl_er(lf)), float(10**_spl_sig(lf))

#  Apply frequency-dependent material properties (P.527-3 + P.2040-3)
def set_materials(sc, freq_hz):
    er_g, sig_g = p527_wet(freq_hz)
    for obj in sc.objects.values():
        mat = getattr(obj, "radio_material", None)
        if mat is None: continue
        mn = mat.name.lower()
        if "terrain" in mn or "ground" in mn or "wet_ground" in mn:
            # P.527-3: always applied manually 
            mat.relative_permittivity, mat.conductivity = er_g, sig_g
        elif freq_hz < 1e9:
            
            if   "metal" in mn: mat.relative_permittivity, mat.conductivity = 1.0, 1e7
            elif "brick" in mn: mat.relative_permittivity, mat.conductivity = 3.91, 0.020
            elif "wood"  in mn: mat.relative_permittivity, mat.conductivity = 1.99, 0.004
            else:               mat.relative_permittivity, mat.conductivity = 5.24, 0.068

#  frequency setter 
def set_freq(sc, freq_hz):
    try:
        sc.frequency = float(freq_hz)
    except Exception:
        # ITU material callbacks raise when the frequency is outside their valid range.
        # Temporarily remove them, set the frequency, then restore them.
        _cbs = {n: getattr(m,"_frequency_update_callback",None)
                for n,m in sc.radio_materials.items()}
        for n,m in sc.radio_materials.items():
            if _cbs[n]: m._frequency_update_callback = None
        try:   sc.frequency = float(freq_hz)
        except Exception: pass
        finally:
            for n,m in sc.radio_materials.items():
                if n in _cbs: m._frequency_update_callback = _cbs[n]

#  Solvers
_rm_solver   = RadioMapSolver()
_path_solver = PathSolver()

#  Convert lat/lon to local UTM coordinates 
def latlon_to_local(lat, lon):
    # lat/lon -> UTM zone 20N, then subtract the SW-corner scene origin
    _t = Transformer.from_crs("EPSG:4326","EPSG:32620",always_xy=True)
    x_utm, y_utm = _t.transform(lon, lat)
    return x_utm - X_ORI, y_utm - Y_ORI

print(f"Antenna dataset     : {len(df)} sectors")
print(f"Scene extent        : {CM_SIZE_X:.0f} m (E) x {CM_SIZE_Y:.0f} m (N)")


## 2. Scene loading <a id='scene'></a>

Loads the Halifax 3D scene from the multi-material XML file in `data/scenes/halifax_peninsula/`. To compare against the mono-concrete scene, set `SCENE_XML = str(SCENE_XML_MONO_CONCRETE)` before loading.
Separate PLY meshes for brick/concrete/metal/wood walls, asphalt/concrete/metal roofs,
and the LiDAR terrain surface. Material properties follow ITU-R P.2040-3 (buildings)
and P.527-3 (terrain).


In [ ]:
scene = load_scene(SCENE_XML)
scene.frequency = 2.8e9   # initial value, overridden by the frequency widget at simulation time
set_materials(scene, float(np.array(scene.frequency).item()))
print(f"Scene: {len(scene.objects)} objects | freq = {float(np.array(scene.frequency).item())/1e9:.2f} GHz")

## 3. Carrier frequency <a id='freq'></a>

Select a frequency band. The antenna count shown next to each option is the number
of real antennas operating on that band in Halifax.
The center frequency is used for ray tracing, material properties, and noise floor.


In [ ]:
# Each entry: (fmin_MHz, fmax_MHz, centre_Hz) 
BANDS = {
    "700 MHz  (LTE/4G)":    (680,  760,   700e6),
    "850 MHz  (LTE/4G)":    (800,  900,   850e6),
    "1700 MHz (LTE/4G)":    (1700, 1780, 1750e6),
    "1900 MHz (LTE/4G)":    (1850, 1990, 1900e6),
    "2100 MHz (LTE/4G)":    (1920, 2170, 2100e6),
    "2600 MHz (LTE/4G)":    (2500, 2700, 2600e6),
    "3500 MHz (5G)":         (3300, 3700, 3500e6),
}
CUSTOM_BAND_KEY = "__custom__"

def count_ant(fmin, fmax):
    return len(df[df.freq_mhz.between(fmin,fmax)]
               .drop_duplicates(subset=["x_local_m","y_local_m","azimuth_deg"]))

def get_real_antennas(band_label, max_tx):
    if band_label == CUSTOM_BAND_KEY:
        return pd.DataFrame(), custom_freq_w.value * 1e6
    fmin, fmax, freq_hz = BANDS[band_label]
    sub = (df[df.freq_mhz.between(fmin, fmax)]
           .drop_duplicates(subset=["x_local_m","y_local_m","azimuth_deg"], keep="first")  # one sector per beam direction
           .head(max_tx)
           .reset_index(drop=True))
    return sub, freq_hz

def get_freq_hz():
    if freq_w.value == CUSTOM_BAND_KEY:
        return custom_freq_w.value * 1e6
    return BANDS[freq_w.value][2]

band_options = [(f"{k}  [{count_ant(v[0],v[1])} antennas]", k)
                for k, v in BANDS.items()]
band_options.append(("Custom frequency", CUSTOM_BAND_KEY))

freq_w = widgets.Dropdown(options=band_options, value="3500 MHz (5G)",
                           description="Band:", style={"description_width":"60px"},
                           layout=widgets.Layout(width="480px"))
max_tx_w = widgets.IntSlider(value=10, min=1, max=30, step=1,
                              description="Max TX:", style={"description_width":"60px"},
                              layout=widgets.Layout(width="340px"))
custom_freq_w = widgets.BoundedFloatText(
    value=2800.0, min=1.0, max=100000.0, step=0.1,
    description="Freq (MHz):",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="260px", display="none"))

out_freq = widgets.Output()
def _refresh_freq(_=None):
    is_custom = (freq_w.value == CUSTOM_BAND_KEY)
    custom_freq_w.layout.display = "" if is_custom else "none"
    with out_freq:
        clear_output(wait=True)
        if is_custom:
            print(f"  Custom frequency: {custom_freq_w.value:.1f} MHz — no real antenna data for this band. Try making custom antennas for this frequency")
            return
        sub, freq_hz = get_real_antennas(freq_w.value, max_tx_w.value)
        if len(sub) == 0:
            print("  No real antennas found for this band. Try making custom antennas for this frequency")
            return
        by_op = sub["licensee"].apply(lambda x: x.split()[0]).value_counts()
        print(f"  {len(sub)} antennas at {freq_hz/1e6:.0f} MHz")
        for op, n in by_op.items():
            print(f"    {op}: {n}")
freq_w.observe(_refresh_freq, names="value")
max_tx_w.observe(_refresh_freq, names="value")
custom_freq_w.observe(_refresh_freq, names="value")
_refresh_freq()
display(widgets.HBox([freq_w, max_tx_w]), custom_freq_w, out_freq)

## 4. TX z override <a id='height'></a>

By default, each antenna is placed at its real height.
Enable the override to force all antennas to the same height.
3GPP power class changes automatically with height:
≥ 20 m → 43 dBm macro, 10–20 m → 38 dBm medium range, < 10 m → 24 dBm small cell.


In [ ]:
height_override_w = widgets.Checkbox(value=False, description="Override all TX z",
                                      layout=widgets.Layout(width="220px"))
height_val_w = widgets.FloatSlider(value=30.0, min=5.0, max=100.0, step=5.0,  # 30 m: typical urban macro site height
                                    description="TX z (m):", disabled=True,
                                    style={"description_width":"90px"},
                                    layout=widgets.Layout(width="380px"))
def _toggle_h(c): height_val_w.disabled = not height_override_w.value
height_override_w.observe(_toggle_h, names="value")
display(widgets.HBox([height_override_w, height_val_w]))


## 5. Transmit power override <a id='power'></a>

By default, each antenna uses the 3GPP power class for its height.
Enable the override to force a fixed TX power for all antennas.


In [ ]:
power_override_w = widgets.Checkbox(value=False, description="Override TX power",
                                     layout=widgets.Layout(width="210px"))
power_val_w = widgets.FloatSlider(value=43.0, min=0.0, max=60.0, step=1.0,
                                   description="P_TX (dBm):", disabled=True,
                                   style={"description_width":"90px"},
                                   layout=widgets.Layout(width="360px"))
power_lbl = widgets.Label()
def _upd_pwr_lbl(_=None):
    power_lbl.value = f"= {10**(power_val_w.value/10)*1e-3:.2f} W"   # dBm to watts
_upd_pwr_lbl()
power_val_w.observe(_upd_pwr_lbl, names="value")
def _toggle_p(c): power_val_w.disabled = not power_override_w.value
power_override_w.observe(_toggle_p, names="value")
display(widgets.VBox([power_override_w, widgets.HBox([power_val_w, power_lbl])]))

## 6. Antenna pattern <a id='pattern'></a>

- **Directional** — Unidirectionnal antenna. Azimuth and tilt from antennas dataset are applied.
- **dipole** — Half-wave dipole (2.15 dBi), toroidal radiation, 360° azimuth.
- **iso** — Isotropic (0 dBi), radiates equally in all directions. Theoretical reference.


In [ ]:
pattern_w = widgets.RadioButtons(
    options=[("Directional", "tr38901"),
             ("Omnidirectional  (half-wave dipole)", "dipole"),
             ("Isotropic  (reference)", "iso")],
    value="tr38901",
    description="Pattern:", style={"description_width":"70px"})
display(pattern_w)
def get_pattern(): return pattern_w.value

## 7. Radio-map plane and ray tracing <a id='rt'></a>

- **RX plane z** - absolute scene elevation of the horizontal radio-map plane, in meters.
  It is not a height above the local terrain. The approximate local height above ground is
  `RX plane z - DEM(x, y)` and therefore changes across the peninsula.
- **Important default** - `55 m` is a practical demo value that keeps the receiver plane above
  many roofs and avoids many ground/building intersections. It is not a true street-level user height.
- **Future user-height mode** - a better street-level workflow would place receivers at
  `DEM(x, y) + user_height`, for example `DEM + 1.5 m`, instead of using one fixed horizontal plane.
- **Edge diffraction** - UTD diffraction around building edges.
- **max_depth** - maximum number of ray interactions with scene surfaces.
- **Rays per TX** - number of random rays launched per transmitter. More rays reduce noise but increase runtime.
- **Cell size** - grid resolution of the radio map. Smaller cells give more detail but increase runtime and memory.


In [ ]:
rx_height_w = widgets.FloatSlider(value=55.0, min=1.5, max=200.0, step=2.5,  # Absolute scene z of the fixed horizontal radio-map plane.
                                   description="RX plane z (m):",
                                   style={"description_width":"110px"},
                                   layout=widgets.Layout(width="420px"))
diff_w    = widgets.Checkbox(value=True,  description="Edge diffraction (UTD)")
diffuse_w = widgets.Checkbox(value=True,  description="Diffuse scattering")
depth_w   = widgets.IntSlider(value=2, min=1, max=12, step=1,
                               description="max_depth:",
                               style={"description_width":"90px"},
                               layout=widgets.Layout(width="380px"))
samples_w = widgets.SelectionSlider(
    options=["50 k","100 k","500 k","1 M","2 M"], value="100 k",
    description="Rays/TX:", style={"description_width":"80px"},
    layout=widgets.Layout(width="380px"))
cellsize_w = widgets.SelectionSlider(
    options=["40 m","80 m","150 m","250 m"], value="150 m",   
    description="Cell size:", style={"description_width":"80px"},
    layout=widgets.Layout(width="380px"))

def get_samples(): return int(float(samples_w.value.replace(" ","").replace("k","e3").replace("M","e6")))
def get_cellsize(): return float(cellsize_w.value.replace(" m",""))

display(widgets.VBox([
    rx_height_w,
    widgets.HBox([diff_w, diffuse_w]),
    depth_w, samples_w, cellsize_w
]))


## 8.  Custom antennas <a id='custom'></a>

Define your own transmitter(s) to test a hypothetical deployment

Enter the **latitude / longitude** of the antenna.
The notebook converts them automatically to local UTM coordinates.
The **azimuth** follows the standard convention: 0° = North, clockwise positive.

Use **Antenna source mode** to choose whether to simulate:
- Real antennas only (real deployment)
- Custom antennas only (your hypothetical deployment)
- Both combined


In [ ]:
#  Antenna source mode 
mode_w = widgets.RadioButtons(
    options=["Real antennas only",
             "Custom antennas only",
             "Real + Custom antennas"],
    value="Real antennas only",
    description="Source:", style={"description_width":"70px"})
display(mode_w)

#  Custom antenna definition form 
_TX_BANDS = list(BANDS.keys())

name_w    = widgets.Text(value="custom_001",  description="Name:",
                          layout=widgets.Layout(width="260px"))
lat_w     = widgets.FloatText(value=44.647,   description="Latitude (N or S):",
                               layout=widgets.Layout(width="220px"))
lon_w     = widgets.FloatText(value=-63.587,  description="Longitude (E or W):",
                               layout=widgets.Layout(width="220px"))
# z_local_m is an absolute scene elevation, not mast height above ground.
# Example: 30 m mast + 45 m terrain elevation -> z_local_m = 75 m.
z_cust_w  = widgets.FloatSlider(value=30.0, min=5.0, max=100.0, step=5.0,
                                 description="TX z (m):",
                                 style={"description_width":"90px"},
                                 layout=widgets.Layout(width="380px"))
az_w      = widgets.FloatSlider(value=0.0, min=0.0, max=359.0, step=5.0,  # 0° = North, 90° = East, clockwise
                                 description="Azimuth (°):",
                                 style={"description_width":"95px"},
                                 layout=widgets.Layout(width="380px"))
tilt_w    = widgets.FloatSlider(value=3.0, min=0.0, max=20.0, step=1.0,
                                 description="Downtilt (°):",
                                 style={"description_width":"95px"},
                                 layout=widgets.Layout(width="380px"))
freq_cust_w = widgets.Dropdown(
    options=[(k, k) for k in _TX_BANDS] + [("Custom frequency", CUSTOM_BAND_KEY)],
    value="3500 MHz (5G)",
    description="Band:",
    style={"description_width":"60px"},
    layout=widgets.Layout(width="360px"))
freq_cust_custom_w = widgets.BoundedFloatText(
    value=2800.0, min=1.0, max=100000.0, step=0.1,
    description="Freq (MHz):",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="260px", display="none"))
def _on_freq_cust_band(_=None):
    freq_cust_custom_w.layout.display = (
        "" if freq_cust_w.value == CUSTOM_BAND_KEY else "none")
freq_cust_w.observe(_on_freq_cust_band, names="value")
pwr_cust_w  = widgets.FloatSlider(value=43.0, min=0.0, max=60.0, step=1.0,
                                   description="Power (dBm):",
                                   style={"description_width":"95px"},
                                   layout=widgets.Layout(width="380px"))
pat_cust_w  = widgets.RadioButtons(
    options=[("Directional", "tr38901"),
             ("Omnidirectional  (half-wave dipole)", "dipole"),
             ("Isotropic  (reference)", "iso")],
    value="tr38901", description="Pattern:", style={"description_width":"70px"})

add_btn   = widgets.Button(description="+ Add antenna", button_style="primary",
                            layout=widgets.Layout(width="160px"))
clear_btn = widgets.Button(description="Clear all", button_style="warning",
                            layout=widgets.Layout(width="120px"))
list_out  = widgets.Output()

custom_txs = []   # list of dicts, one per custom TX

def _show_list():
    with list_out:
        clear_output(wait=True)
        if not custom_txs:
            print("  No custom antennas defined.")
            return
        print(f"  {'#':<3} {'Name':<14} {'Lat':>8} {'Lon':>9} {'z':>5} {'Az':>5} "
              f"{'Tilt':>5} {'Band':<22} {'P_TX':>6} {'Pattern'}")
        print("  " + "-"*95)
        for i, tx in enumerate(custom_txs):
            print(f"  {i:<3} {tx['name']:<14} {tx['lat']:>8.4f} {tx['lon']:>9.4f} "
                  f"{tx['z_local_m']:>5.1f} {tx['azimuth_deg']:>5.1f} {tx['tilt_deg']:>5.1f} "
                  f"{tx['band']:<22} {tx['power_dbm']:>6.1f} {tx['pattern']}")


def validate_custom_tx(name, x_l, y_l, z_l, freq_mhz, power_dbm, azimuth_deg):
    """Return validation issues for a custom transmitter definition."""
    issues = []
    clean_name = str(name).strip()
    if not clean_name:
        issues.append("name is empty")
    if clean_name in {str(tx.get("name", "")).strip() for tx in custom_txs}:
        issues.append("name already exists")
    if not (np.isfinite(x_l) and np.isfinite(y_l)):
        issues.append("latitude/longitude could not be projected")
    elif not (0.0 <= x_l <= CM_SIZE_X and 0.0 <= y_l <= CM_SIZE_Y):
        issues.append("projected position is outside the scene extent")
    if not np.isfinite(z_l) or z_l <= 0.0:
        issues.append("TX z must be a positive absolute scene elevation")
    if not np.isfinite(freq_mhz) or not (1.0 <= freq_mhz <= 100000.0):
        issues.append("frequency must be between 1 MHz and 100 GHz")
    if not np.isfinite(power_dbm) or not (0.0 <= power_dbm <= 80.0):
        issues.append("TX power is outside the allowed 0-80 dBm range")
    for tx in custom_txs:
        same_position = abs(tx["x_local_m"] - x_l) < 0.01 and abs(tx["y_local_m"] - y_l) < 0.01
        same_frequency = abs(tx["freq_mhz"] - freq_mhz) < 0.001
        same_azimuth = abs(tx["azimuth_deg"] - azimuth_deg) < 0.1
        if same_position and same_frequency and same_azimuth:
            issues.append("duplicate position/frequency/azimuth already exists")
            break
    return issues

def _add_tx(_):
    x_l, y_l = latlon_to_local(lat_w.value, lon_w.value)
    if freq_cust_w.value == CUSTOM_BAND_KEY:
        freq_hz   = freq_cust_custom_w.value * 1e6
        freq_mhz  = freq_cust_custom_w.value
        band_name = f"Custom {freq_mhz:.1f} MHz"
    else:
        fmin, fmax, freq_hz = BANDS[freq_cust_w.value]
        freq_mhz = (fmin + fmax) / 2.0
        band_name = freq_cust_w.value
    issues = validate_custom_tx(name_w.value, x_l, y_l, z_cust_w.value,
                                freq_mhz, pwr_cust_w.value, az_w.value)
    if issues:
        with list_out:
            clear_output(wait=True)
            print("Cannot add custom antenna:")
            for issue in issues:
                print(f"  - {issue}")
        return

    custom_txs.append({
        "name":        name_w.value.strip(),
        "lat":         lat_w.value,
        "lon":         lon_w.value,
        "x_local_m":   x_l,
        "y_local_m":   y_l,
        "z_local_m":   z_cust_w.value,
        "azimuth_deg": az_w.value,
        "tilt_deg":    tilt_w.value,
        "band":        band_name,
        "freq_mhz":    freq_mhz,
        "freq_hz":     freq_hz,
        "power_dbm":   pwr_cust_w.value,
        "pattern":     pat_cust_w.value,
        "licensee":    "Custom",
    })
    # Auto-increment name
    base = name_w.value.rstrip("0123456789_")
    name_w.value = f"{base}_{len(custom_txs)+1:03d}"
    _show_list()

def _clear_txs(_):
    custom_txs.clear()
    _show_list()

add_btn.on_click(_add_tx)
clear_btn.on_click(_clear_txs)

_show_list()
display(widgets.VBox([
    widgets.HBox([name_w, lat_w, lon_w]),
    z_cust_w, az_w, tilt_w,
    widgets.HBox([freq_cust_w, freq_cust_custom_w]),
    pwr_cust_w, pat_cust_w,
    widgets.HBox([add_btn, clear_btn]),
    list_out,
]))


## 9. All parameters — Run simulation <a id='run'></a>

Review the parameter summary below and click **▶ Run Coverage Map** to start.
The simulation places real and/or custom antennas in the Sionna scene and computes
the coverage map with `RadioMapSolver`.


In [ ]:
run_btn = widgets.Button(description="▶  Run Coverage Map",
                          button_style="success",
                          layout=widgets.Layout(width="240px", height="40px"))
run_out  = widgets.Output()
sim      = {}   # simulation results stored here for downstream cells

def _build_tx_list():
    "Returns list of TX dicts based on mode_w selection."
    txs = []
    mode = mode_w.value
    if mode in ("Real antennas only","Real + Custom antennas"):
        sub, _freq_sub = get_real_antennas(freq_w.value, max_tx_w.value)
        for row_idx, r in sub.iterrows():
            z   = height_val_w.value if height_override_w.value else float(r["z_local_m"])
            pwr = power_val_w.value  if power_override_w.value  else get_tx_power_dbm(z)
            txs.append({"name": f"ised_{row_idx:03d}",
                        "x_local_m": float(r["x_local_m"]),
                        "y_local_m": float(r["y_local_m"]),
                        "z_local_m": float(z),
                        "azimuth_deg": float(r["azimuth_deg"]),
                        "tilt_deg":    float(r["tilt_deg"]),
                        "power_dbm":  pwr,
                        "pattern":    get_pattern(),
                        "freq_hz":    get_freq_hz(),
                        "licensee":   str(r["licensee"])})
    if mode in ("Custom antennas only","Real + Custom antennas"):
        for ct in custom_txs:
            z   = ct["z_local_m"]
            pwr = power_val_w.value if power_override_w.value else ct["power_dbm"]
            txs.append({"name":      ct["name"],
                        "x_local_m": ct["x_local_m"],
                        "y_local_m": ct["y_local_m"],
                        "z_local_m": z,
                        "azimuth_deg": ct["azimuth_deg"],
                        "tilt_deg":    ct["tilt_deg"],
                        "power_dbm":  pwr,
                        "pattern":    ct["pattern"],
                        "freq_hz":    ct["freq_hz"]})
    return txs

summary_out = widgets.Output()
def _refresh_summary(_=None):
    with summary_out:
        clear_output(wait=True)
        txs = _build_tx_list()
        freq_hz  = get_freq_hz()
        band_lbl = (f"Custom {freq_hz/1e6:.1f} MHz" if freq_w.value == CUSTOM_BAND_KEY
                    else freq_w.value)
        print(f"Band          : {band_lbl}  ({freq_hz/1e6:.0f} MHz)")
        print(f"Mode          : {mode_w.value}")
        print(f"TX count      : {len(txs)}")
        _height_str = f"fixed absolute z = {height_val_w.value:.1f} m" if height_override_w.value else "real antenna absolute z"
        print(f"TX z source   : {_height_str}")
        if txs and not power_override_w.value:
            _pwrs = sorted({round(t["power_dbm"]) for t in txs})
            _pwr_str = f"{_pwrs[0]} dBm" if len(_pwrs) == 1 else f"{_pwrs[0]}-{_pwrs[-1]} dBm"
        else:
            _pwr_str = f"fixed {power_val_w.value:.1f} dBm" if power_override_w.value else "3GPP class from TX z"
        print(f"TX power      : {_pwr_str}")
        _pat_label = next((lbl for lbl, val in pattern_w.options if val == get_pattern()), get_pattern())
        print(f"Pattern       : {_pat_label}")
        print(f"RX plane z    : {rx_height_w.value:.1f} m (fixed horizontal plane)")
        print(f"Diffraction   : {'ON' if diff_w.value else 'OFF'}  |  Diffuse: {'ON' if diffuse_w.value else 'OFF'}")
        print(f"max_depth     : {depth_w.value}")
        print(f"Rays per TX   : {samples_w.value} ({get_samples():.2e})")
        print(f"Cell size     : {cellsize_w.value}")
        if custom_txs:
            print(f"Custom TX     : {len(custom_txs)} defined")

for w in [freq_w, max_tx_w, mode_w, height_override_w, height_val_w,
          power_override_w, power_val_w, pattern_w, rx_height_w,
          diff_w, diffuse_w, depth_w, samples_w, cellsize_w,
          custom_freq_w]:
    w.observe(_refresh_summary, names="value")
_refresh_summary()

def _run(_):
    with run_out:
        clear_output(wait=True)
        import time; gc.collect()
        txs = _build_tx_list()
        if not txs:
            print("ERROR: no TX to simulate (check mode selection and custom antennas).")
            return
        _refresh_summary()
        freq_hz = get_freq_hz()
        cs  = get_cellsize()
        spt = get_samples()

        print("Scene configuration")
        print("-------------------")
        set_freq(scene, freq_hz)
        set_materials(scene, freq_hz)

        for name in list(scene.transmitters): scene.remove(name)
        for name in list(scene.receivers):    scene.remove(name)
        pat = get_pattern()
        scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                                      vertical_spacing=0.5, horizontal_spacing=0.5,
                                      pattern=pat, polarization="V")
        scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                                      vertical_spacing=0.5, horizontal_spacing=0.5,
                                      pattern="iso", polarization="V")

        for tx in txs:
            az  = math.radians(tx["azimuth_deg"])
            tlt = math.radians(tx["tilt_deg"])
            x, y, z = tx["x_local_m"], tx["y_local_m"], tx["z_local_m"]
            scene.add(Transmitter(
                name=tx["name"],
                position=[x, y, z],
                look_at=[x + 500*math.sin(az), y + 500*math.cos(az), z + 500*math.sin(tlt)],   
                power_dbm=tx["power_dbm"],
            ))
        print(f"TX placed     : {len(scene.transmitters)}")
        print("Solver        : RadioMapSolver")

        t0 = time.time()
        rm = _rm_solver(
            scene,
            cell_size=(cs, cs),
            center=[CM_SIZE_X/2, CM_SIZE_Y/2, rx_height_w.value],
            orientation=[0., 0., 0.],
            size=[CM_SIZE_X, CM_SIZE_Y],
            samples_per_tx=int(spt),
            max_depth=depth_w.value,
            los=True,
            specular_reflection=True,
            edge_diffraction=diff_w.value,
            diffuse_reflection=diffuse_w.value,
        )
        dt = time.time() - t0

        rss    = np.array(rm.path_gain)          # (N_TX, H, W) — linear path gain, dimensionless
        P_max  = max(tx["power_dbm"] for tx in txs)
        P_ref  = power_val_w.value if power_override_w.value else P_max   # reference TX power in dBm
        best   = rss.max(axis=0)   # best-server: highest path gain across all TX at each pixel
        rsrp   = P_ref + 10*np.log10(np.clip(best, 1e-20, None))   # RSRP = P_TX_dBm + 10·log10(G_best)

        sim.update({"rss":rm, "rss_np":rss, "rsrp":rsrp, "freq_hz":freq_hz,
                    "txs":txs, "P_ref":P_ref, "cs":cs, "radiomap":rm})

        print("Simulation result")
        print("-----------------")
        print(f"Runtime       : {dt:.0f} s")
        valid = rss.max(axis=0) > 0
        if not valid.any():
            print("WARNING: no valid cells — TX outside scene bounds or placed underground.")
            for tx in txs:
                print(f"  {tx['name']}: x={tx['x_local_m']:.1f} m, "
                      f"y={tx['y_local_m']:.1f} m, z={tx['z_local_m']:.1f} m")
            print(f"  Scene XY bounds: x=[0, {CM_SIZE_X:.0f}] m, y=[0, {CM_SIZE_Y:.0f}] m")
            print("  Check: z must be above terrain height at that location.")
            return
        print(f"Valid cells   : {100*valid.mean():.1f}%")
        print(f"RSRP range    : [{rsrp[valid].min():.0f}, {rsrp[valid].max():.0f}] dBm")
        print("Next          : run the result plotting cells below.")

run_btn.on_click(_run)

display(summary_out, run_btn, run_out)


## 10. Coverage map results <a id='results'></a>

Coverage map over the Halifax peninsula.
TX positions shown as triangles
(white = real antennas, red = custom antennas).


In [ ]:
if not sim:
    print("No results yet — run the simulation first.")
else:
    rsrp  = sim["rsrp"]
    txs   = sim["txs"]
    fhz   = sim["freq_hz"]
    P_ref = sim["P_ref"]
    H, W  = rsrp.shape
    geo   = build_geo_mask(H, W)
    masked = np.where(geo, rsrp, np.nan)

    fig, ax = plt.subplots(figsize=(11, 9))
    fig.patch.set_facecolor("#0d0d1a"); ax.set_facecolor("#0d0d1a")
    cm_ = plt.cm.plasma.copy(); cm_.set_bad("#0d0d1a")
    im  = ax.imshow(masked, origin="lower",
                    extent=[0,CM_SIZE_X,0,CM_SIZE_Y],
                    cmap=cm_, vmin=-140, vmax=-60, aspect="equal")   
    cb = plt.colorbar(im, ax=ax, label="RSRP (dBm)", fraction=0.035)
    cb.ax.yaxis.label.set_color("white"); cb.ax.tick_params(colors="white")

    # Colour markers by operator: Custom=red, Bell=blue, Rogers=pink, Bragg=green
    OP_COLORS = {
        "Bell":   "#4FC3F7",
        "Rogers": "#E118D4",
        "Bragg":  "#2EC833",
        "Custom": "#FF1744",
    }
    op_seen = set()
    for tx in txs:
        lic = tx.get("licensee", "ISED")
        op  = next((k for k in OP_COLORS if k in lic), None)
        col  = OP_COLORS.get(op, "#FFFFFF")
        mark = "s" if lic == "Custom" else "^"   # square for custom TX, triangle for real antennas
        label = op if op and op not in op_seen else None
        if label: op_seen.add(op)
        ax.plot(tx["x_local_m"], tx["y_local_m"], marker=mark, color=col,
                ms=10, markeredgecolor="black", markeredgewidth=0.8,
                zorder=10, label=label)

    ax.set_title(f"RSRP — {fhz/1e6:.0f} MHz | {len(txs)} TX | P_ref={P_ref:.0f} dBm | RX plane z={rx_height_w.value:.0f} m",
                 color="white", fontsize=12)
    ax.set_xlabel("East (m)", color="white")
    ax.set_ylabel("North (m)", color="white")
    ax.tick_params(colors="white", labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor("#444")

    ax.legend(loc="lower right", fontsize=8,
              facecolor="#222", labelcolor="white", framealpha=0.8,
              markerscale=0.9)

    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, f"custom_rsrp_{fhz/1e6:.0f}MHz.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show(); print(f"Saved: {out_path}")


### SINR map (co-channel)

Co-channel SINR computed as `G_signal / (G_interf + N0/P_TX)`,
where only TX sharing the same frequency channel contribute to interference.


In [ ]:
if not sim or sim["rss_np"].shape[0] < 2:
    print("SINR requires at least 2 TX.")
else:
    rss_np = sim["rss_np"]; P_ref = sim["P_ref"]; fhz = sim["freq_hz"]
    N_TX, H, W = rss_np.shape
    BW_Hz    = 20e6                           # assumed channel bandwidth
    P_TX_W   = 10**(P_ref/10)*1e-3               # dBm to watts
    N0_W     = 10**(-174/10)*1e-3*BW_Hz          # thermal noise: kT·BW at room temperature 
    N0_ratio = N0_W / P_TX_W                     # noise normalised to TX power

    # Best-server SINR: at each pixel, select the TX that gives the highest SINR
    sinr = np.full((H,W), -np.inf)
    for k in range(N_TX):
        G_s  = rss_np[k]
        G_i  = rss_np.sum(axis=0) - G_s   # co-channel interference from all other TX
        s_db = 10*np.log10(np.clip(G_s/(G_i+N0_ratio), 1e-20, None))
        sinr = np.maximum(sinr, s_db)

    geo = build_geo_mask(H, W)
    sinr_m = np.where(geo, sinr, np.nan)

    ZONES = [(20,100,"#2ecc71","Excellent > 20 dB"),
             (10, 20,"#f1c40f","Good 10–20 dB"),
             ( 0, 10,"#e67e22","Marginal 0–10 dB"),
             (-30, 0,"#e74c3c","Interfered < 0 dB")]

    from matplotlib.colors import to_rgba
    quality = np.full((H,W,4), np.nan)
    for lo,hi,col,_ in ZONES:
        m = geo & (sinr >= lo) & (sinr < hi)
        quality[m] = list(to_rgba(col)[:3]) + [0.9]

    fig, (a1,a2) = plt.subplots(1, 2, figsize=(16,7))
    fig.patch.set_facecolor("#0d0d1a")
    for ax in (a1,a2): ax.set_facecolor("#111")

    cm_s = plt.cm.RdYlGn.copy(); cm_s.set_bad("#111")
    im1 = a1.imshow(sinr_m, origin="lower", extent=[0,CM_SIZE_X,0,CM_SIZE_Y],
                    cmap=cm_s, vmin=-10, vmax=30, aspect="equal")
    cb1 = plt.colorbar(im1, ax=a1, label="SINR (dB)", fraction=0.035)
    cb1.ax.yaxis.label.set_color("white"); cb1.ax.tick_params(colors="white")
    a1.set_title(f"SINR — {fhz/1e6:.0f} MHz", color="white")
    a1.tick_params(colors="white", labelsize=8)

    a2.imshow(quality, origin="lower", extent=[0,CM_SIZE_X,0,CM_SIZE_Y], aspect="equal")
    a2.legend(handles=[Patch(facecolor=c,label=l) for _,_,c,l in ZONES],
              loc="lower right", fontsize=8, facecolor="#222",
              labelcolor="white", framealpha=0.8)
    a2.set_title("Network quality zones", color="white")
    a2.tick_params(colors="white", labelsize=8)

    for ax in (a1,a2):
        for tx in sim["txs"]:
            col = "#FF4444" if tx.get("licensee")=="Custom" else "white"
            ax.plot(tx["x_local_m"],tx["y_local_m"],marker="^",color=col,
                    ms=8,markeredgecolor="black",markeredgewidth=0.8,zorder=10)
        ax.set_xlabel("Local East (m)", color="white")

    plt.tight_layout(rect=[0,0,1,0.95])
    plt.suptitle(f"Co-channel SINR | {fhz/1e6:.0f} MHz | {N_TX} TX",
                 color="white", fontsize=11)
    out_s = os.path.join(OUT_DIR, f"custom_sinr_{fhz/1e6:.0f}MHz.png")
    plt.savefig(out_s, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

    t = geo.sum()
    print(f"Excellent (>20 dB) : {100*(geo&(sinr>20)).sum()/t:.1f}%")
    print(f"Good (10-20 dB)    : {100*(geo&(sinr>10)&(sinr<=20)).sum()/t:.1f}%")
    print(f"Marginal (0-10 dB) : {100*(geo&(sinr>0 )&(sinr<=10)).sum()/t:.1f}%")
    print(f"Interfered (< 0dB) : {100*(geo&(sinr<=0)).sum()/t:.1f}%")


## 11. 3D scene preview <a id='preview'></a>

Opens the Sionna 3D interactive viewer.

- **Show ray paths** — runs `PathSolver` on a small set of TX-RX pairs and overlays
  the computed paths (LoS, reflected, diffracted) on the scene.

Use mouse-drag to orbit, scroll to zoom. Requires a display.


In [ ]:
show_paths_w = widgets.Checkbox(value=False,
    description="Show ray paths (runs PathSolver — slow)",
    layout=widgets.Layout(width="380px"))
n_rx_w = widgets.IntSlider(value=4, min=1, max=20, step=1,
    description="RX per TX:", style={"description_width":"80px"},
    layout=widgets.Layout(width="300px"), disabled=True)
rx_dist_w = widgets.FloatSlider(value=300.0, min=50.0, max=1000.0, step=50.0,
    description="RX dist (m):", style={"description_width":"90px"},
    layout=widgets.Layout(width="340px"), disabled=True)

def _toggle_paths(c):
    n_rx_w.disabled   = not show_paths_w.value
    rx_dist_w.disabled = not show_paths_w.value
show_paths_w.observe(_toggle_paths, names="value")
display(widgets.VBox([show_paths_w, widgets.HBox([n_rx_w, rx_dist_w])]))

# Launch preview 
# The TX are already placed from the last simulation run.
# Re-sync scene if needed.
if sim:
    set_freq(scene, sim["freq_hz"])
    set_materials(scene, sim["freq_hz"])
    for name in list(scene.transmitters): scene.remove(name)
    for name in list(scene.receivers):    scene.remove(name)
    for tx in sim["txs"]:
        az  = math.radians(tx["azimuth_deg"])
        tlt = math.radians(tx["tilt_deg"])
        x, y, z = tx["x_local_m"], tx["y_local_m"], tx["z_local_m"]
        scene.add(Transmitter(
            name=tx["name"],
            position=[x, y, z],
            look_at=[x+500*math.sin(az), y+500*math.cos(az), z+500*math.sin(tlt)],
            power_dbm=tx["power_dbm"],
        ))

if show_paths_w.value and sim:
    print("Computing ray paths with PathSolver...")
    scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                                  vertical_spacing=0.5, horizontal_spacing=0.5,
                                  pattern="iso", polarization="V")
    for tx in list(sim["txs"])[:3]:   # limit to first 3 TX for speed
        x, y, z = tx["x_local_m"], tx["y_local_m"], tx["z_local_m"]
        for k in range(n_rx_w.value):
            angle = 2*math.pi * k / n_rx_w.value
            scene.add(Receiver(
                name=f"rx_{tx['name']}_{k}",
                position=[x + rx_dist_w.value*math.cos(angle),
                          y + rx_dist_w.value*math.sin(angle),
                          rx_height_w.value],
            ))
    paths = _path_solver(scene, max_depth=depth_w.value,
                          los=True, specular_reflection=True,
                          edge_diffraction=diff_w.value)
    print(f"Paths computed. Launching 3D preview...")
    scene.preview(paths=paths, show_devices=True)
elif sim and sim.get("radiomap"):
    scene.preview(show_devices=True)
else:
    scene.preview(show_devices=True)

## 12. 3D ray tracing visualisation <a id='raytrace3d'></a>

Launches the Sionna 3D interactive viewer with computed ray paths overlaid
on the urban scene geometry. TX antennas are taken from the last simulation
(or from the current widget settings). RX receivers are placed in a fan
pattern centred on the TX azimuth direction.

**Path colours in the Sionna viewer:**
- Green  = Line-of-Sight (LoS)
- Blue   = Specularly reflected
- Red    = Diffracted (UTD)
- Orange = Diffusely scattered

**Widgets:**
- *RX per TX* - number of receivers placed around each transmitter.
- *RX distance* - radius of the RX fan, in meters.
- *RX z* - absolute scene elevation of the receivers. It is not automatically terrain-following.
- *Max TX* - transmitter-count limit for the 3D viewer.
- *max_depth* - maximum number of ray interactions per path.


In [ ]:
#  Antenna source mode for 3D (independent from coverage map)
mode_3d_w = widgets.ToggleButtons(
    options=["Real antennas only", "Custom antennas only", "Real + Custom antennas"],
    value="Real antennas only",
    description="Source:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="620px"),
)

#  Compact custom antenna form (feeds the same custom_txs as section 8)
c3d_lat_w   = widgets.FloatText(value=44.647, description="Lat:",
    layout=widgets.Layout(width="180px"))
c3d_lon_w   = widgets.FloatText(value=-63.587, description="Lon:",
    layout=widgets.Layout(width="200px"))
c3d_z_w     = widgets.FloatSlider(value=30.0, min=5.0, max=100.0, step=5.0,
    description="TX z (m):", style={"description_width":"85px"},
    layout=widgets.Layout(width="340px"))
c3d_az_w    = widgets.FloatSlider(value=0.0, min=0.0, max=359.0, step=5.0,
    description="Azimuth (°):", style={"description_width":"85px"},
    layout=widgets.Layout(width="340px"))
c3d_band_w  = widgets.Dropdown(
    options=[(k, k) for k in BANDS.keys()] + [("Custom frequency", CUSTOM_BAND_KEY)],
    value="3500 MHz (5G)", description="Band:",
    style={"description_width":"50px"}, layout=widgets.Layout(width="360px"))
c3d_pwr_w   = widgets.FloatSlider(value=43.0, min=0.0, max=60.0, step=1.0,
    description="Power (dBm):", style={"description_width":"85px"},
    layout=widgets.Layout(width="340px"))
c3d_add_btn  = widgets.Button(description="+ Add custom TX", button_style="primary",
    layout=widgets.Layout(width="160px"))
c3d_clear_btn = widgets.Button(description="Clear all", button_style="warning",
    layout=widgets.Layout(width="110px"))
c3d_list_out = widgets.Output()

def _c3d_refresh():
    with c3d_list_out:
        clear_output(wait=True)
        if not custom_txs:
            print("  No custom TX defined.")
            return
        print(f"  {len(custom_txs)} custom TX:")
        for i, tx in enumerate(custom_txs):
            print(f"    [{i}] {tx['name']}  lat={tx['lat']:.4f} lon={tx['lon']:.4f}"
                  f"  z={tx['z_local_m']:.0f}m  az={tx['azimuth_deg']:.0f}°  {tx['band']}  {tx['power_dbm']:.0f} dBm")

def _c3d_add(_):
    x_l, y_l = latlon_to_local(c3d_lat_w.value, c3d_lon_w.value)
    if c3d_band_w.value == CUSTOM_BAND_KEY:
        freq_hz = get_freq_hz()
        band_name = f"Custom {freq_hz/1e6:.1f} MHz"
    else:
        _, _, freq_hz = BANDS[c3d_band_w.value]
        band_name = c3d_band_w.value
    name = f"c3d_{len(custom_txs)+1:03d}"
    issues = validate_custom_tx(name, x_l, y_l, c3d_z_w.value,
                                freq_hz/1e6, c3d_pwr_w.value, c3d_az_w.value)
    if issues:
        with c3d_list_out:
            clear_output(wait=True)
            print("Cannot add custom antenna:")
            for issue in issues:
                print(f"  - {issue}")
        return

    custom_txs.append({
        "name": name,
        "lat": c3d_lat_w.value, "lon": c3d_lon_w.value,
        "x_local_m": x_l, "y_local_m": y_l, "z_local_m": c3d_z_w.value,
        "azimuth_deg": c3d_az_w.value, "tilt_deg": 3.0,
        "band": band_name, "freq_mhz": freq_hz/1e6, "freq_hz": freq_hz,
        "power_dbm": c3d_pwr_w.value, "pattern": get_pattern(), "licensee": "Custom",
    })
    _c3d_refresh()

def _c3d_clear(_):
    custom_txs.clear()
    _c3d_refresh()

c3d_add_btn.on_click(_c3d_add)
c3d_clear_btn.on_click(_c3d_clear)
_c3d_refresh()

custom_3d_box = widgets.VBox([
    widgets.HTML("<b>Add custom antenna</b>"),
    widgets.HBox([c3d_lat_w, c3d_lon_w]),
    c3d_z_w, c3d_az_w, c3d_band_w, c3d_pwr_w,
    widgets.HBox([c3d_add_btn, c3d_clear_btn]),
    c3d_list_out,
], layout=widgets.Layout(display="none", border="1px solid #ccc",
                         padding="8px", margin="4px 0 8px 0"))

def _on_mode_3d(change):
    show = change["new"] in ("Custom antennas only", "Real + Custom antennas")
    custom_3d_box.layout.display = "" if show else "none"

mode_3d_w.observe(_on_mode_3d, names="value")

#  RX placement + ray tracing widgets 
n_rx_3d_w    = widgets.IntSlider(value=4, min=1, max=12, step=1,
    description="RX per TX:", style={"description_width":"100px"},
    layout=widgets.Layout(width="360px"))
rx_dist_3d_w = widgets.FloatSlider(value=300.0, min=50.0, max=1500.0, step=50.0,
    description="RX dist (m):", style={"description_width":"100px"},
    layout=widgets.Layout(width="380px"))
rx_h_3d_w    = widgets.FloatSlider(value=1.5, min=1.5, max=55.0, step=2.5,
    description="RX z (m):", style={"description_width":"105px"},
    layout=widgets.Layout(width="380px"))
max_tx_3d_w  = widgets.IntSlider(value=3, min=1, max=10, step=1,
    description="Max TX:", style={"description_width":"100px"},
    layout=widgets.Layout(width="360px"))
depth_3d_w   = widgets.IntSlider(value=2, min=1, max=10, step=1,
    description="max_depth:", style={"description_width":"100px"},
    layout=widgets.Layout(width="360px"))

rt3d_btn = widgets.Button(description="▶  Launch 3D Ray Tracing",
    button_style="warning",
    layout=widgets.Layout(width="260px", height="40px"))
rt3d_out = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<b>Antenna source</b>"),
    mode_3d_w,
    custom_3d_box,
    widgets.HTML("<b>Receivers</b>"),
    widgets.HBox([n_rx_3d_w, rx_dist_3d_w]),
    rx_h_3d_w,
    widgets.HTML("<b>Ray tracing</b>"),
    widgets.HBox([max_tx_3d_w, depth_3d_w]),
    rt3d_btn,
    rt3d_out,
]))


def _launch_3d(_):
    _rt3d_paths = [None]
    with rt3d_out:
        clear_output(wait=True)
        import time; gc.collect()

        # Build TX list according to mode_3d_w (independent from coverage map mode)
        txs_full = []
        mode_3d  = mode_3d_w.value
        if mode_3d in ("Real antennas only", "Real + Custom antennas"):
            sub, _ = get_real_antennas(freq_w.value, max_tx_w.value)
            for row_idx, r in sub.iterrows():
                z   = height_val_w.value if height_override_w.value else float(r["z_local_m"])
                pwr = power_val_w.value  if power_override_w.value  else get_tx_power_dbm(z)
                txs_full.append({
                    "name":        f"ised_{row_idx:03d}",
                    "x_local_m":   float(r["x_local_m"]),
                    "y_local_m":   float(r["y_local_m"]),
                    "z_local_m":   z,
                    "azimuth_deg": float(r["azimuth_deg"]),
                    "tilt_deg":    float(r["tilt_deg"]),
                    "power_dbm":   pwr,
                    "pattern":     get_pattern(),
                    "freq_hz":     get_freq_hz(),
                    "licensee":    str(r["licensee"]),
                })
        if mode_3d in ("Custom antennas only", "Real + Custom antennas"):
            for ct in custom_txs:
                pwr = power_val_w.value if power_override_w.value else ct["power_dbm"]
                txs_full.append({
                    "name":        ct["name"],
                    "x_local_m":   ct["x_local_m"],
                    "y_local_m":   ct["y_local_m"],
                    "z_local_m":   ct["z_local_m"],
                    "azimuth_deg": ct["azimuth_deg"],
                    "tilt_deg":    ct["tilt_deg"],
                    "power_dbm":   pwr,
                    "pattern":     ct.get("pattern", get_pattern()),
                    "freq_hz":     ct["freq_hz"],
                    "licensee":    "Custom",
                })

        if not txs_full:
            print("No TX — configure antennas in section 8 or select a band first.")
            return

        txs_rt  = txs_full[:max_tx_3d_w.value]
        freq_hz = txs_rt[0]["freq_hz"]
        pat_rt  = get_pattern()

        print(f"3D Ray Tracing | {freq_hz/1e6:.0f} MHz | {len(txs_rt)} TX | "
              f"{n_rx_3d_w.value} RX/TX | dist={rx_dist_3d_w.value:.0f} m | "
              f"RX z={rx_h_3d_w.value:.1f} m | depth={depth_3d_w.value}")

        # Configure scene 
        set_freq(scene, freq_hz)
        set_materials(scene, freq_hz)

        for name in list(scene.transmitters): scene.remove(name)
        for name in list(scene.receivers):    scene.remove(name)

        scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
            vertical_spacing=0.5, horizontal_spacing=0.5,
            pattern=pat_rt, polarization="V")
        scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
            vertical_spacing=0.5, horizontal_spacing=0.5,
            pattern="iso", polarization="V")

        # Place TX
        for tx in txs_rt:
            az  = math.radians(tx["azimuth_deg"])
            tlt = math.radians(tx["tilt_deg"])
            x, y, z = tx["x_local_m"], tx["y_local_m"], tx["z_local_m"]
            scene.add(Transmitter(
                name=tx["name"],
                position=[x, y, z],
                look_at=[x + 500 * math.sin(az),
                         y + 500 * math.cos(az),
                         z + 500 * math.sin(tlt)],
                power_dbm=tx["power_dbm"],
            ))

        #  Place RX in a fan centred on each TX azimuth 
        n_rx = n_rx_3d_w.value
        for tx in txs_rt:
            x, y     = tx["x_local_m"], tx["y_local_m"]
            base_az  = math.radians(tx.get("azimuth_deg", 0.0))
            for k in range(n_rx):
                if n_rx > 1:
                    spread = math.pi * k / (n_rx - 1) - math.pi / 2
                else:
                    spread = 0.0
                angle = base_az + spread
                rx_x  = x + rx_dist_3d_w.value * math.sin(angle)
                rx_y  = y + rx_dist_3d_w.value * math.cos(angle)
                rx_x  = max(10.0, min(CM_SIZE_X - 10.0, rx_x))
                rx_y  = max(10.0, min(CM_SIZE_Y - 10.0, rx_y))
                scene.add(Receiver(
                    name=f"rx_{tx['name']}_{k:02d}",
                    position=[rx_x, rx_y, rx_h_3d_w.value],
                ))

        n_tx = len(scene.transmitters)
        n_rx_total = len(scene.receivers)
        print(f"Placed: {n_tx} TX  |  {n_rx_total} RX")

        # Run PathSolver 
        print(f"PathSolver running (depth={depth_3d_w.value})...", end=" ", flush=True)
        t0 = time.time()
        paths = _path_solver(
            scene,
            max_depth=depth_3d_w.value,
            los=True,
            specular_reflection=True,
            edge_diffraction=diff_w.value,
            diffuse_reflection=diffuse_w.value,
        )
        dt = time.time() - t0
        print(f"done in {dt:.0f} s")

        #  Path statistics 
        try:
            tau     = np.array(paths.tau)
            n_valid = int((tau > 1e-12).sum())
            print(f"Valid ray paths found: {n_valid}")
        except Exception:
            pass

        #  Launch 3D preview 
        print("Opening 3D viewer...")
        _rt3d_paths[0] = paths

    # scene.preview must be outside Output() for WebGL canvas to render
    if _rt3d_paths[0] is not None:
        scene.preview(paths=_rt3d_paths[0], show_devices=True)

rt3d_btn.on_click(_launch_3d)
